# Local CPU Maxmix Entropy-Dynamics Characterization

This notebook reads saved maxmix purification entropy-contour products, validates all selected campaigns together, writes combined `S/ln2` tables, and then analyzes each observable by system size. Different system sizes are never mixed in the same figure; within a fixed system size, protocol, `nshell`, and `dw_truncation` variants are grouped as subplots/curves.

## Configurations And Domain-Wall Geometry

| campaign | Nx | Ny | nshell values | protocols / samples | cycles | init state | dtype | dw_truncation | alpha_1 / alpha_2 | status |
|---|---:|---:|---|---|---:|---|---|---|---|---|
| N16x32_C100_dwtrunc0 | 16 | 32 | 1, 2 | perfect_correction / 10; postselect / 1 | 100 | maxmix | complex128 | False | 1 / 30 | active |
| N16x32_C100_dwtrunc1 | 16 | 32 | 1, 2 | perfect_correction / 10; postselect / 1 | 100 | maxmix | complex128 | True | 1 / 30 | comparison |
| N30x40_C100_dwtrunc1_nsh1 | 30 | 40 | 1 | perfect_correction / 25; postselect / 1 | 100 | maxmix | complex128 | True | 1 / 30 | single_nshell |

Domain-wall geometry:

| Nx | half | w=floor(0.2*Nx) | DW_loc | topological alpha_1 slab | trivial alpha_2 regions |
|---:|---:|---:|---|---|---|
| 16 | 8 | 3 | [5, 11] | x=5..11 | x=0..4 and x=12..15 |
| 30 | 15 | 6 | [9, 21] | x=9..21 | x=0..8 and x=22..29 |

The domain walls extend along the full periodic `y` direction. Entropy dataframes and rendered figures use `S/ln2` units. The raw `.npz` files remain in natural-log units and are validated before normalization.


## 1. Setup

In [ ]:
from __future__ import annotations

import json
from pathlib import Path
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Image, Markdown, Video, display

plt.rcParams.update({
    'figure.dpi': 130,
    'savefig.dpi': 180,
    'axes.grid': True,
    'grid.alpha': 0.25,
    'axes.spines.top': False,
    'axes.spines.right': False,
})

BUNDLE_NAME = 'colab_small_system_testing'
DEFAULT_CAMPAIGN_MANIFESTS = [
    'purification_entropy_contours_maxmix/campaigns/N16x32_C100_dwtrunc0/campaign_manifest.json',
    'purification_entropy_contours_maxmix/campaigns/N16x32_C100_dwtrunc1/campaign_manifest.json',
    'purification_entropy_contours_maxmix/campaigns/N30x40_C100_dwtrunc1_nsh1/campaign_manifest.json',
]
CAMPAIGN_MANIFEST_OVERRIDE = None
REGISTRY_PATH = 'purification_entropy_contours_maxmix/campaigns_manifest.json'
COMPARISON_ID = 'N16x32_C100_dwtrunc_comparison'
FINAL_CYCLE = 100
ENTROPY_NORM = float(np.log(2.0))
ENTROPY_EPS = 1e-12
TOTAL_ENTROPY_OFFSET_OVER_LN2 = 2.0
SPATIAL_ENTROPY_OFFSET_OVER_LN2 = 1.0
ENTROPY_UNITS = 'S_over_ln2'

candidates = []
for base in [Path.cwd(), *Path.cwd().parents]:
    candidates.extend([base, base / BUNDLE_NAME])

seen = set()
BUNDLE_ROOT = None
for candidate in candidates:
    candidate = candidate.resolve()
    if candidate in seen:
        continue
    seen.add(candidate)
    if (candidate / 'gpu_data' / 'purification_entropy_contours_maxmix').exists():
        BUNDLE_ROOT = candidate
        break
if BUNDLE_ROOT is None:
    raise FileNotFoundError(f'Could not locate {BUNDLE_NAME}/gpu_data/purification_entropy_contours_maxmix')

GPU_DATA_ROOT = BUNDLE_ROOT / 'gpu_data'
DATA_ROOT = GPU_DATA_ROOT / 'purification_entropy_contours_maxmix'
ANALYSIS_ROOT = BUNDLE_ROOT / 'analysis_outputs' / 'purification_entropy_contours_maxmix_cpu'
OUTPUT_ROOT = ANALYSIS_ROOT / COMPARISON_ID
FIG_ROOT = OUTPUT_ROOT / 'figures'
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
FIG_ROOT.mkdir(parents=True, exist_ok=True)

print(f'[setup] bundle root: {BUNDLE_ROOT}')
print(f'[setup] data root: {DATA_ROOT}')
print(f'[setup] output root: {OUTPUT_ROOT}')

## 2. Shared Helpers

In [ ]:
def load_json(path: Path):
    with Path(path).open('r', encoding='utf-8') as fh:
        return json.load(fh)


def write_json_atomic(path: Path, payload) -> None:
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + '.tmp')
    with tmp.open('w', encoding='utf-8') as fh:
        json.dump(payload, fh, indent=2, sort_keys=True)
    tmp.replace(path)


def save_npz_atomic(path: Path, **arrays) -> None:
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + '.tmp')
    with tmp.open('wb') as fh:
        np.savez_compressed(fh, **arrays)
    tmp.replace(path)


def save_csv_atomic(df: pd.DataFrame, path: Path) -> None:
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + '.tmp')
    df.to_csv(tmp, index=False)
    tmp.replace(path)


def rel_to_bundle(path: Path) -> str:
    return str(Path(path).resolve().relative_to(BUNDLE_ROOT.resolve()))


def rel_to_gpu_data(path: Path) -> str:
    return str(Path(path).resolve().relative_to(GPU_DATA_ROOT.resolve()))


def resolve_manifest_path(path_like) -> Path:
    path = Path(path_like)
    candidates = [path]
    if not path.is_absolute():
        candidates.extend([GPU_DATA_ROOT / path, BUNDLE_ROOT / path])
    for candidate in candidates:
        if candidate.exists():
            return candidate.resolve()
    raise FileNotFoundError(f'Campaign manifest not found: {path_like}')


def resolve_campaign_manifests() -> list[Path]:
    if CAMPAIGN_MANIFEST_OVERRIDE:
        values = CAMPAIGN_MANIFEST_OVERRIDE
        if isinstance(values, (str, Path)):
            values = [values]
    else:
        values = list(DEFAULT_CAMPAIGN_MANIFESTS)
    manifest_paths = [resolve_manifest_path(value) for value in values]
    if len(set(manifest_paths)) != len(manifest_paths):
        raise ValueError(f'Duplicate campaign manifests: {manifest_paths}')
    return manifest_paths


def case_id_from_summary(summary: dict) -> str:
    return f"N{summary['Nx']}x{summary['Ny']}_nsh{summary['nshell']}_{summary['protocol']}"


def dw_label(value: bool) -> str:
    return 'dw_trunc1' if bool(value) else 'dw_trunc0'


def show_saved_image(path: Path, title: str | None = None, note: str | None = None) -> None:
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(path)
    if title:
        display(Markdown(f'**{title}**'))
    if note:
        display(Markdown(note))
    display(Image(filename=str(path)))


def show_saved_video(path: Path, title: str | None = None, note: str | None = None) -> None:
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(path)
    if title:
        display(Markdown(f'**{title}**'))
    if note:
        display(Markdown(note))
    display(Video(filename=str(path), embed=False, html_attributes='controls'))


def require_ffmpeg() -> None:
    from matplotlib.animation import writers
    try:
        import imageio_ffmpeg
        plt.rcParams['animation.ffmpeg_path'] = imageio_ffmpeg.get_ffmpeg_exe()
    except Exception:
        pass
    if not writers.is_available('ffmpeg'):
        raise RuntimeError('Saving MP4 requires ffmpeg or imageio-ffmpeg.')

def system_label(nx: int, ny: int) -> str:
    return f'N{int(nx)}x{int(ny)}'


def available_system_sizes(records: list[dict]) -> list[tuple[int, int]]:
    return sorted({(int(rec['summary']['Nx']), int(rec['summary']['Ny'])) for rec in records})


def records_for_system(records: list[dict], nx: int, ny: int) -> list[dict]:
    return [rec for rec in records if int(rec['summary']['Nx']) == int(nx) and int(rec['summary']['Ny']) == int(ny)]


def outside_legend(ax, **kwargs):
    handles, labels = ax.get_legend_handles_labels()
    if handles:
        ax.legend(
            handles,
            labels,
            loc='center left',
            bbox_to_anchor=(1.02, 0.5),
            borderaxespad=0.0,
            frameon=True,
            **kwargs,
        )


def remove_stale_figures(names: list[str]) -> None:
    for name in names:
        path = FIG_ROOT / name
        if path.exists():
            path.unlink()
            print(f'[cleanup] removed stale figure {path.name}')


## 3. Load And Validate All Selected Campaigns

In [ ]:
REQUIRED_NPZ_KEYS = {
    'entropy_contour_avg', 'entropy_contour_std', 'entropy_contour_sem',
    'entropy_contour_sum', 'entropy_contour_sumsq', 'entropy_total_mean',
    'entropy_total_std', 'entropy_total_sem', 'entropy_total_sum',
    'entropy_total_sumsq', 'sample_counts_by_cycle', 'cycles', 'config_json',
}


def load_campaign_records(manifest_path: Path) -> tuple[dict, list[dict], dict]:
    manifest = load_json(manifest_path)
    campaign_id = manifest.get('campaign_id', manifest_path.parent.name)
    results = manifest.get('results', [])
    if not results:
        raise ValueError(f'{campaign_id}: no results in campaign manifest')

    campaign_status = str(manifest.get('status', 'complete'))
    is_single_nshell = campaign_status == 'single_nshell'
    if len(results) not in {2, 4}:
        raise ValueError(f'{campaign_id}: expected 2 or 4 runs, got {len(results)}')
    if len(results) == 2 and not is_single_nshell:
        raise ValueError(f'{campaign_id}: 2-run campaigns must be marked status=single_nshell')

    records = []
    validation_report = {
        'campaign_id': campaign_id,
        'campaign_manifest': rel_to_bundle(manifest_path),
        'status': campaign_status,
        'runs': {},
    }
    seen = set()
    for entry in results:
        npz_path = GPU_DATA_ROOT / entry['npz_path_relative']
        summary_path = GPU_DATA_ROOT / entry['summary_path_relative']
        if not npz_path.exists():
            raise FileNotFoundError(npz_path)
        if not summary_path.exists():
            raise FileNotFoundError(summary_path)
        summary = load_json(summary_path)
        case_id = case_id_from_summary(summary)
        run_key = f'{campaign_id}/{case_id}'
        if run_key in seen:
            raise ValueError(f'{campaign_id}: duplicate case id {case_id}')
        seen.add(run_key)

        nx = int(summary['Nx'])
        ny = int(summary['Ny'])
        cycles_expected = int(summary['cycles'])
        samples = int(summary['samples'])
        protocol = str(summary['protocol'])
        if summary.get('init_mode') != 'maxmix':
            raise ValueError(f'{run_key}: expected init_mode=maxmix, got {summary.get("init_mode")}')
        if summary.get('dtype_resolved') != 'complex128':
            raise ValueError(f'{run_key}: expected dtype_resolved=complex128, got {summary.get("dtype_resolved")}')
        if protocol not in {'perfect_correction', 'postselect'}:
            raise ValueError(f'{run_key}: unexpected protocol={protocol}')

        with np.load(npz_path, allow_pickle=False) as data:
            missing = sorted(REQUIRED_NPZ_KEYS - set(data.files))
            if missing:
                raise KeyError(f'{npz_path} missing keys {missing}')
            cycles = np.asarray(data['cycles'], dtype=np.int64)
            S_xy = np.asarray(data['entropy_contour_avg'], dtype=np.float64)
            S_total = np.asarray(data['entropy_total_mean'], dtype=np.float64)
            S_total_sem = np.asarray(data['entropy_total_sem'], dtype=np.float64)
            S_total_std = np.asarray(data['entropy_total_std'], dtype=np.float64)
            expected_shape = (cycles_expected + 1, nx, ny)
            if S_xy.shape != expected_shape:
                raise ValueError(f'{npz_path}: expected contour shape {expected_shape}, got {S_xy.shape}')
            if not np.array_equal(cycles, np.arange(cycles_expected + 1)):
                raise ValueError(f'{npz_path}: unexpected cycle values')
            numeric = [np.asarray(data[k]) for k in REQUIRED_NPZ_KEYS if k not in {'cycles', 'config_json'}]
            if not all(np.all(np.isfinite(arr)) for arr in numeric):
                raise FloatingPointError(f'{npz_path}: non-finite values')
            config = json.loads(str(np.asarray(data['config_json']).item()))
            if config.get('init_mode') != 'maxmix' or config.get('dtype_resolved') != 'complex128':
                raise ValueError(f'{run_key}: config_json metadata mismatch')
            total_from_contour = S_xy.sum(axis=(1, 2))
            total_err = float(np.max(np.abs(total_from_contour - S_total)))
            S_yint = S_xy.sum(axis=2)
            yint_err = float(np.max(np.abs(S_yint.sum(axis=1) - S_total)))
            sum_err = float(np.max(np.abs(np.asarray(data['entropy_contour_sum']) - samples * S_xy)))
            if total_err > 1e-8 or yint_err > 1e-8 or sum_err > 1e-8:
                raise ValueError(f'{run_key}: validation mismatch total={total_err:g}, yint={yint_err:g}, sum={sum_err:g}')
            if samples == 1:
                sem_max = float(max(
                    np.max(np.abs(data['entropy_contour_std'])),
                    np.max(np.abs(data['entropy_contour_sem'])),
                    np.max(np.abs(S_total_std)),
                    np.max(np.abs(S_total_sem)),
                ))
                if sem_max > 1e-12:
                    raise ValueError(f'{run_key}: expected zero SEM/std for one sample, got {sem_max:g}')
            if np.any(np.diff(S_total) > 1e-8):
                warnings.warn(f'{run_key}: total entropy is not strictly monotone nonincreasing within tolerance')

        x_keep = np.arange(0, nx // 2 + 1, dtype=np.int64)
        record = {
            'campaign_id': campaign_id,
            'case_id': case_id,
            'summary': summary,
            'npz_path': npz_path,
            'summary_path': summary_path,
            'cycles': cycles,
            'S_xy': S_xy,
            'S_total': S_total,
            'S_total_sem': S_total_sem,
            'S_total_std': S_total_std,
            'S_yint': S_yint,
            'x_keep': x_keep,
            'S_yint_half': S_yint[:, x_keep],
            'dw_label': dw_label(bool(summary['dw_truncation'])),
            'single_nshell_campaign': is_single_nshell,
        }
        records.append(record)
        validation_report['runs'][case_id] = {
            'npz_path': rel_to_bundle(npz_path),
            'summary_path': rel_to_bundle(summary_path),
            'max_total_from_contour_error': total_err,
            'max_yintegrated_total_error': yint_err,
            'max_sum_consistency_error': sum_err,
            'min_contour_value': float(np.min(S_xy)),
            'max_contour_value': float(np.max(S_xy)),
            'shape': list(S_xy.shape),
        }

    records.sort(key=lambda r: (int(r['summary']['Nx']), int(r['summary']['Ny']), int(r['summary']['nshell']), str(r['summary']['protocol']), r['dw_label']))
    return manifest, records, validation_report

manifest_paths = resolve_campaign_manifests()
print('[load] campaign manifests:')
for path in manifest_paths:
    print(f'  {path}')

all_manifests = []
records = []
validation_report = {'campaigns': {}}
for path in manifest_paths:
    manifest, campaign_records, campaign_validation = load_campaign_records(path)
    all_manifests.append(manifest)
    records.extend(campaign_records)
    validation_report['campaigns'][campaign_validation['campaign_id']] = campaign_validation

if not records:
    raise RuntimeError('No records loaded')
write_json_atomic(OUTPUT_ROOT / 'validation_report.json', validation_report)
print(f'[load] loaded {len(records)} runs across {len(all_manifests)} campaigns')

## 4. Build Combined Tables

In [ ]:
run_rows = []
summary_rows = []
curve_rows = []
yint_rows = []
for rec in records:
    s = rec['summary']
    cycles = rec['cycles']
    S = rec['S_total'] / ENTROPY_NORM
    S_sem = rec['S_total_sem'] / ENTROPY_NORM
    S_std = rec['S_total_std'] / ENTROPY_NORM
    x_keep = rec['x_keep']
    S_yint_half = rec['S_yint_half'] / ENTROPY_NORM
    S_abs_yint_minus_log2_half = np.abs(S_yint_half - SPATIAL_ENTROPY_OFFSET_OVER_LN2)
    run_rows.append({
        'campaign_id': rec['campaign_id'],
        'case_id': rec['case_id'],
        'Nx': int(s['Nx']),
        'Ny': int(s['Ny']),
        'nshell': int(s['nshell']),
        'protocol': s['protocol'],
        'samples': int(s['samples']),
        'cycles': int(s['cycles']),
        'dw_truncation': bool(s['dw_truncation']),
        'dw_label': rec['dw_label'],
        'single_nshell_campaign': bool(rec['single_nshell_campaign']),
        'entropy_units': ENTROPY_UNITS,
        'npz_path': rel_to_bundle(rec['npz_path']),
        'summary_path': rel_to_bundle(rec['summary_path']),
    })

    def value_at(cycle):
        idx = np.where(cycles == int(cycle))[0]
        return float(S[idx[0]]) if len(idx) else np.nan

    summary_rows.append({
        'campaign_id': rec['campaign_id'],
        'case_id': rec['case_id'],
        'Nx': int(s['Nx']),
        'Ny': int(s['Ny']),
        'nshell': int(s['nshell']),
        'protocol': s['protocol'],
        'samples': int(s['samples']),
        'dw_truncation': bool(s['dw_truncation']),
        'dw_label': rec['dw_label'],
        'entropy_units': ENTROPY_UNITS,
        'S0_over_ln2': value_at(0),
        'S10_over_ln2': value_at(10),
        'S25_over_ln2': value_at(25),
        'S50_over_ln2': value_at(50),
        'S100_over_ln2': value_at(100),
        'S_final_over_ln2': float(S[-1]),
        'S0_minus_2log2_over_ln2': value_at(0) - TOTAL_ENTROPY_OFFSET_OVER_LN2,
        'S10_minus_2log2_over_ln2': value_at(10) - TOTAL_ENTROPY_OFFSET_OVER_LN2,
        'S25_minus_2log2_over_ln2': value_at(25) - TOTAL_ENTROPY_OFFSET_OVER_LN2,
        'S50_minus_2log2_over_ln2': value_at(50) - TOTAL_ENTROPY_OFFSET_OVER_LN2,
        'S100_minus_2log2_over_ln2': value_at(100) - TOTAL_ENTROPY_OFFSET_OVER_LN2,
        'S_final_minus_2log2_over_ln2': float(S[-1] - TOTAL_ENTROPY_OFFSET_OVER_LN2),
        'abs_S0_minus_2log2_over_ln2': abs(value_at(0) - TOTAL_ENTROPY_OFFSET_OVER_LN2),
        'abs_S10_minus_2log2_over_ln2': abs(value_at(10) - TOTAL_ENTROPY_OFFSET_OVER_LN2),
        'abs_S25_minus_2log2_over_ln2': abs(value_at(25) - TOTAL_ENTROPY_OFFSET_OVER_LN2),
        'abs_S50_minus_2log2_over_ln2': abs(value_at(50) - TOTAL_ENTROPY_OFFSET_OVER_LN2),
        'abs_S100_minus_2log2_over_ln2': abs(value_at(100) - TOTAL_ENTROPY_OFFSET_OVER_LN2),
        'abs_S_final_minus_2log2_over_ln2': float(abs(S[-1] - TOTAL_ENTROPY_OFFSET_OVER_LN2)),
        'S_final_over_S0': float(S[-1] / S[0]),
        'min_total_over_ln2': float(np.min(S)),
        'min_total_minus_2log2_over_ln2': float(np.min(S - TOTAL_ENTROPY_OFFSET_OVER_LN2)),
        'min_abs_total_minus_2log2_over_ln2': float(np.min(np.abs(S - TOTAL_ENTROPY_OFFSET_OVER_LN2))),
        'final_total_sem_over_ln2': float(S_sem[-1]),
        'max_total_sem_over_ln2': float(np.max(S_sem)),
    })
    for cyc, val, sem, std in zip(cycles, S, S_sem, S_std):
        curve_rows.append({
            'campaign_id': rec['campaign_id'],
            'case_id': rec['case_id'],
            'Nx': int(s['Nx']),
            'Ny': int(s['Ny']),
            'nshell': int(s['nshell']),
            'protocol': s['protocol'],
            'samples': int(s['samples']),
            'dw_truncation': bool(s['dw_truncation']),
            'dw_label': rec['dw_label'],
            'entropy_units': ENTROPY_UNITS,
            'cycle': int(cyc),
            'total_entropy_over_ln2': float(val),
            'total_entropy_minus_2log2_over_ln2': float(val - TOTAL_ENTROPY_OFFSET_OVER_LN2),
            'total_entropy_abs_minus_2log2_over_ln2': float(abs(val - TOTAL_ENTROPY_OFFSET_OVER_LN2)),
            'total_entropy_std_over_ln2': float(std),
            'total_entropy_sem_over_ln2': float(sem),
            'total_entropy_minus_2log2_std_over_ln2': float(std),
            'total_entropy_minus_2log2_sem_over_ln2': float(sem),
        })
    for t_idx, cyc in enumerate(cycles):
        for xi, x in enumerate(x_keep):
            yint_rows.append({
                'campaign_id': rec['campaign_id'],
                'case_id': rec['case_id'],
                'Nx': int(s['Nx']),
                'Ny': int(s['Ny']),
                'nshell': int(s['nshell']),
                'protocol': s['protocol'],
                'samples': int(s['samples']),
                'dw_truncation': bool(s['dw_truncation']),
                'dw_label': rec['dw_label'],
                'entropy_units': ENTROPY_UNITS,
                'cycle': int(cyc),
                'x': int(x),
                'y_integrated_entropy_over_ln2': float(S_yint_half[t_idx, xi]),
                'y_integrated_entropy_spatial_minus_log2_over_ln2': float(
                    S_yint_half[t_idx, xi] - SPATIAL_ENTROPY_OFFSET_OVER_LN2
                ),
                'y_integrated_abs_spatial_minus_log2_over_ln2': float(
                    S_abs_yint_minus_log2_half[t_idx, xi]
                ),
            })

run_index = pd.DataFrame(run_rows).sort_values(['Nx', 'Ny', 'nshell', 'protocol', 'dw_label']).reset_index(drop=True)
total_entropy_summary = pd.DataFrame(summary_rows).sort_values(['Nx', 'Ny', 'nshell', 'protocol', 'dw_label']).reset_index(drop=True)
total_entropy_curves = pd.DataFrame(curve_rows).sort_values(['Nx', 'Ny', 'nshell', 'protocol', 'dw_label', 'cycle']).reset_index(drop=True)
y_integrated_entropy_curves = pd.DataFrame(yint_rows).sort_values(['Nx', 'Ny', 'nshell', 'protocol', 'dw_label', 'cycle', 'x']).reset_index(drop=True)

save_csv_atomic(run_index, OUTPUT_ROOT / 'run_index.csv')
save_csv_atomic(total_entropy_summary, OUTPUT_ROOT / 'total_entropy_summary.csv')
save_csv_atomic(total_entropy_curves, OUTPUT_ROOT / 'total_entropy_curves.csv')
save_csv_atomic(y_integrated_entropy_curves, OUTPUT_ROOT / 'y_integrated_entropy_curves.csv')

display(Markdown('### Run index'))
display(run_index)
display(Markdown('### Total entropy summary (`S/ln2`)'))
display(total_entropy_summary)


## 5. Save Derived Arrays

In [ ]:
# Stacked arrays are written only when all loaded campaigns share the same cycle grid and x_keep.
case_ids = np.asarray([rec['case_id'] for rec in records])
campaign_ids = np.asarray([rec['campaign_id'] for rec in records])
dw_labels = np.asarray([rec['dw_label'] for rec in records])
protocols = np.asarray([rec['summary']['protocol'] for rec in records])
nshell_values = np.asarray([int(rec['summary']['nshell']) for rec in records], dtype=np.int64)
Nx_values = np.asarray([int(rec['summary']['Nx']) for rec in records], dtype=np.int64)
Ny_values = np.asarray([int(rec['summary']['Ny']) for rec in records], dtype=np.int64)
cycles_ref = records[0]['cycles']
x_keep_ref = records[0]['x_keep']
if all(np.array_equal(rec['cycles'], cycles_ref) and np.array_equal(rec['x_keep'], x_keep_ref) for rec in records):
    save_npz_atomic(
        OUTPUT_ROOT / 'derived_arrays.npz',
        case_id=case_ids,
        campaign_id=campaign_ids,
        dw_label=dw_labels,
        protocol=protocols,
        nshell=nshell_values,
        Nx=Nx_values,
        Ny=Ny_values,
        cycles=cycles_ref,
        x_keep=x_keep_ref,
        total_entropy_mean_over_ln2=np.stack([rec['S_total'] / ENTROPY_NORM for rec in records], axis=0),
        total_entropy_minus_2log2_mean_over_ln2=np.stack([
            rec['S_total'] / ENTROPY_NORM - TOTAL_ENTROPY_OFFSET_OVER_LN2 for rec in records
        ], axis=0),
        total_entropy_abs_minus_2log2_mean_over_ln2=np.stack([
            np.abs(rec['S_total'] / ENTROPY_NORM - TOTAL_ENTROPY_OFFSET_OVER_LN2) for rec in records
        ], axis=0),
        total_entropy_sem_over_ln2=np.stack([rec['S_total_sem'] / ENTROPY_NORM for rec in records], axis=0),
        total_entropy_minus_2log2_sem_over_ln2=np.stack([rec['S_total_sem'] / ENTROPY_NORM for rec in records], axis=0),
        total_entropy_abs_minus_2log2_sem_over_ln2=np.stack([rec['S_total_sem'] / ENTROPY_NORM for rec in records], axis=0),
        y_integrated_entropy_half_over_ln2=np.stack([rec['S_yint_half'] / ENTROPY_NORM for rec in records], axis=0),
        y_integrated_entropy_spatial_minus_log2_half_over_ln2=np.stack([
            rec['S_yint_half'] / ENTROPY_NORM - SPATIAL_ENTROPY_OFFSET_OVER_LN2
            for rec in records
        ], axis=0),
        y_integrated_abs_spatial_minus_log2_half_over_ln2=np.stack([
            np.abs(rec['S_yint_half'] / ENTROPY_NORM - SPATIAL_ENTROPY_OFFSET_OVER_LN2)
            for rec in records
        ], axis=0),
        entropy_units=np.asarray(ENTROPY_UNITS),
        config_json=np.asarray(json.dumps({'source_campaigns': [m.get('campaign_id') for m in all_manifests]}, sort_keys=True)),
    )
    print('[arrays] wrote derived_arrays.npz')
else:
    stale_path = OUTPUT_ROOT / 'derived_arrays.npz'
    if stale_path.exists():
        stale_path.unlink()
        print('[arrays] removed stale derived_arrays.npz from a previous same-grid analysis')
    print('[arrays] skipped stacked derived_arrays.npz because cycle/x grids differ')

## 6. Total Entropy Dynamics

In [ ]:
matched_records = [rec for rec in records if int(rec['summary']['Nx']) == 16 and int(rec['summary']['Ny']) == 32]
plot_records = records
colors = {'dw_trunc0': 'tab:blue', 'dw_trunc1': 'tab:red'}
linestyles = {'perfect_correction': '-', 'postselect': '--'}

remove_stale_figures([
    'total_entropy_vs_cycle_linear_by_dwtrunc.png',
    'total_entropy_vs_cycle_log_by_dwtrunc.png',
    'total_entropy_delta_dw1_minus_dw0.png',
    'total_entropy_delta_dw1_minus_dw0_N16x32.png',
    'dw_trunc_total_entropy_overlay_and_delta.png',
])
for pattern in [
    'total_entropy_vs_cycle_N*_linear.png',
    'total_entropy_vs_cycle_N*_log.png',
    'total_entropy_vs_cycle_N*_semilog.png',
    'total_entropy_cross_size_nsh*_dw_trunc*_semilog.png',
    'total_entropy_minus_2log2_vs_cycle_N*.png',
    'total_entropy_abs_minus_2log2_vs_cycle_N*.png',
    'total_entropy_minus_2log2_cross_size_nsh*_dw_trunc*.png',
    'total_entropy_abs_minus_2log2_cross_size_nsh*_dw_trunc*.png',
]:
    for stale_path in sorted(FIG_ROOT.glob(pattern)):
        stale_path.unlink()
        print(f'[cleanup] removed stale figure {stale_path.name}')
for stale_output in [
    'dw_trunc_total_entropy_delta.csv',
    'dw_trunc_y_integrated_entropy_delta.csv',
    'dw_trunc_total_entropy_curves.csv',
    'dw_trunc_total_entropy_summary.csv',
    'dw_trunc_y_integrated_entropy_curves.csv',
    'comparison_manifest.json',
]:
    stale_path = OUTPUT_ROOT / stale_output
    if stale_path.exists():
        stale_path.unlink()
        print(f'[cleanup] removed stale output {stale_output}')

for nx, ny in available_system_sizes(plot_records):
    system_records = records_for_system(plot_records, nx, ny)
    nshell_values = sorted({int(rec['summary']['nshell']) for rec in system_records})
    display(Markdown(f'### Total entropy dynamics: N{nx}x{ny} (log-log)'))
    fig, axes = plt.subplots(
        1,
        len(nshell_values),
        figsize=(7.2 * len(nshell_values), 4.5),
        constrained_layout=True,
        squeeze=False,
    )
    axes = axes.ravel()
    for ax, nshell in zip(axes, nshell_values):
        for rec in system_records:
            s = rec['summary']
            if int(s['nshell']) != nshell:
                continue
            sub = total_entropy_curves[
                (total_entropy_curves['Nx'] == nx)
                & (total_entropy_curves['Ny'] == ny)
                & (total_entropy_curves['nshell'] == nshell)
                & (total_entropy_curves['protocol'] == s['protocol'])
                & (total_entropy_curves['dw_label'] == rec['dw_label'])
            ].sort_values('cycle')
            sub = sub[sub['cycle'] > 0]
            if sub.empty:
                continue
            y = sub['total_entropy_over_ln2'].to_numpy(dtype=np.float64)
            err = sub['total_entropy_sem_over_ln2'].to_numpy(dtype=np.float64)
            y_plot = np.maximum(y, ENTROPY_EPS)
            label = f"{s['protocol']} {rec['dw_label']}"
            ax.plot(
                sub['cycle'],
                y_plot,
                color=colors.get(rec['dw_label'], 'black'),
                ls=linestyles.get(s['protocol'], '-'),
                lw=1.5,
                label=label,
            )
            if int(s['samples']) > 1:
                lower = np.maximum(y - err, ENTROPY_EPS)
                upper = np.maximum(y + err, ENTROPY_EPS)
                ax.fill_between(sub['cycle'], lower, upper, color=colors.get(rec['dw_label'], 'black'), alpha=0.12)
        ax.set_title(f'N{nx}x{ny}, nshell={nshell}')
        ax.set_xlabel('cycle')
        ax.set_ylabel('S/ln2')
        ax.set_xscale('log')
        ax.set_yscale('log')
        outside_legend(ax, fontsize=8)
    path = FIG_ROOT / f'total_entropy_vs_cycle_N{nx}x{ny}_loglog.png'
    fig.savefig(path, bbox_inches='tight')
    plt.close(fig)
    show_saved_image(path, f'Total entropy vs cycle, N{nx}x{ny} (log-log)')

for nx, ny in available_system_sizes(plot_records):
    system_records = records_for_system(plot_records, nx, ny)
    nshell_values = sorted({int(rec['summary']['nshell']) for rec in system_records})
    display(Markdown(f'### Absolute total entropy deviation from 2 log 2: N{nx}x{ny} (log-log)'))
    fig, axes = plt.subplots(
        1,
        len(nshell_values),
        figsize=(7.2 * len(nshell_values), 4.5),
        constrained_layout=True,
        squeeze=False,
    )
    axes = axes.ravel()
    for ax, nshell in zip(axes, nshell_values):
        for rec in system_records:
            s = rec['summary']
            if int(s['nshell']) != nshell:
                continue
            sub = total_entropy_curves[
                (total_entropy_curves['Nx'] == nx)
                & (total_entropy_curves['Ny'] == ny)
                & (total_entropy_curves['nshell'] == nshell)
                & (total_entropy_curves['protocol'] == s['protocol'])
                & (total_entropy_curves['dw_label'] == rec['dw_label'])
            ].sort_values('cycle')
            sub = sub[sub['cycle'] > 0]
            if sub.empty:
                continue
            signed = sub['total_entropy_minus_2log2_over_ln2'].to_numpy(dtype=np.float64)
            err = sub['total_entropy_minus_2log2_sem_over_ln2'].to_numpy(dtype=np.float64)
            y = np.maximum(np.abs(signed), ENTROPY_EPS)
            label = f"{s['protocol']} {rec['dw_label']}"
            ax.plot(
                sub['cycle'],
                y,
                color=colors.get(rec['dw_label'], 'black'),
                ls=linestyles.get(s['protocol'], '-'),
                lw=1.5,
                label=label,
            )
            if int(s['samples']) > 1:
                lower_signed = signed - err
                upper_signed = signed + err
                lower = np.where(
                    (lower_signed <= 0.0) & (upper_signed >= 0.0),
                    0.0,
                    np.minimum(np.abs(lower_signed), np.abs(upper_signed)),
                )
                upper = np.maximum(np.abs(lower_signed), np.abs(upper_signed))
                ax.fill_between(
                    sub['cycle'],
                    np.maximum(lower, ENTROPY_EPS),
                    np.maximum(upper, ENTROPY_EPS),
                    color=colors.get(rec['dw_label'], 'black'),
                    alpha=0.12,
                )
        ax.set_title(f'N{nx}x{ny}, nshell={nshell}')
        ax.set_xlabel('cycle')
        ax.set_ylabel(r'$|S - 2\log 2|/\log 2$')
        ax.set_xscale('log')
        ax.set_yscale('log')
        outside_legend(ax, fontsize=8)
    path = FIG_ROOT / f'total_entropy_abs_minus_2log2_vs_cycle_N{nx}x{ny}_loglog.png'
    fig.savefig(path, bbox_inches='tight')
    plt.close(fig)
    show_saved_image(path, f'Absolute total entropy deviation from 2 log 2 vs cycle, N{nx}x{ny} (log-log)')


In [ ]:
# Cross-size total entropy overlay for the shared nshell/dw_truncation setting.
cross_size_systems = [(16, 32), (30, 40)]
cross_size_nshell = 1
cross_size_dw_label = 'dw_trunc1'
cross_size_protocols = ['perfect_correction', 'postselect']
system_colors = {(16, 32): 'tab:blue', (30, 40): 'tab:green'}
protocol_linestyles = {'perfect_correction': '-', 'postselect': '--'}

display(Markdown(
    f'### Cross-size total entropy comparison: nshell={cross_size_nshell}, {cross_size_dw_label}'
))

fig, ax = plt.subplots(figsize=(8.0, 5.0), constrained_layout=True)
plotted = 0
for nx, ny in cross_size_systems:
    for protocol in cross_size_protocols:
        sub = total_entropy_curves[
            (total_entropy_curves['Nx'] == nx)
            & (total_entropy_curves['Ny'] == ny)
            & (total_entropy_curves['nshell'] == cross_size_nshell)
            & (total_entropy_curves['dw_label'] == cross_size_dw_label)
            & (total_entropy_curves['protocol'] == protocol)
        ].sort_values('cycle')
        if sub.empty:
            print(f'[cross-size] missing N{nx}x{ny}, nshell={cross_size_nshell}, {cross_size_dw_label}, {protocol}')
            continue
        sub = sub[sub['cycle'] > 0]
        if sub.empty:
            print(f'[cross-size] no positive-cycle points for N{nx}x{ny}, nshell={cross_size_nshell}, {cross_size_dw_label}, {protocol}')
            continue

        y = sub['total_entropy_over_ln2'].to_numpy(dtype=np.float64)
        err = sub['total_entropy_sem_over_ln2'].to_numpy(dtype=np.float64)
        y_plot = np.maximum(y, ENTROPY_EPS)
        color = system_colors.get((nx, ny), 'black')
        label = f'N{nx}x{ny} {protocol}'
        ax.plot(
            sub['cycle'],
            y_plot,
            color=color,
            ls=protocol_linestyles.get(protocol, '-'),
            lw=1.8,
            label=label,
        )

        if int(sub['samples'].iloc[0]) > 1:
            lower = np.maximum(y - err, ENTROPY_EPS)
            upper = np.maximum(y + err, ENTROPY_EPS)
            ax.fill_between(sub['cycle'], lower, upper, color=color, alpha=0.12)
        plotted += 1

if plotted == 0:
    raise RuntimeError(
        f'No matching cross-size curves for nshell={cross_size_nshell}, {cross_size_dw_label}'
    )

ax.set_title(f'Total entropy vs cycle, nshell={cross_size_nshell}, {cross_size_dw_label}')
ax.set_xlabel('cycle')
ax.set_ylabel('S/ln2')
ax.set_yscale('log')
ax.set_xscale('log')
outside_legend(ax, fontsize=8)

path = FIG_ROOT / f'total_entropy_cross_size_nsh{cross_size_nshell}_{cross_size_dw_label}_loglog.png'
fig.savefig(path, bbox_inches='tight')
plt.close(fig)
show_saved_image(
    path,
    f'Total entropy cross-size comparison, nshell={cross_size_nshell}, {cross_size_dw_label} (log-log)',
)

display(Markdown(
    f'### Cross-size absolute total entropy deviation from 2 log 2: nshell={cross_size_nshell}, {cross_size_dw_label}'
))

fig, ax = plt.subplots(figsize=(8.0, 5.0), constrained_layout=True)
plotted = 0
for nx, ny in cross_size_systems:
    for protocol in cross_size_protocols:
        sub = total_entropy_curves[
            (total_entropy_curves['Nx'] == nx)
            & (total_entropy_curves['Ny'] == ny)
            & (total_entropy_curves['nshell'] == cross_size_nshell)
            & (total_entropy_curves['dw_label'] == cross_size_dw_label)
            & (total_entropy_curves['protocol'] == protocol)
        ].sort_values('cycle')
        if sub.empty:
            print(f'[cross-size shifted] missing N{nx}x{ny}, nshell={cross_size_nshell}, {cross_size_dw_label}, {protocol}')
            continue
        sub = sub[sub['cycle'] > 0]
        if sub.empty:
            print(f'[cross-size shifted] no positive-cycle points for N{nx}x{ny}, nshell={cross_size_nshell}, {cross_size_dw_label}, {protocol}')
            continue

        signed = sub['total_entropy_minus_2log2_over_ln2'].to_numpy(dtype=np.float64)
        err = sub['total_entropy_minus_2log2_sem_over_ln2'].to_numpy(dtype=np.float64)
        y = np.maximum(np.abs(signed), ENTROPY_EPS)
        color = system_colors.get((nx, ny), 'black')
        label = f'N{nx}x{ny} {protocol}'
        ax.plot(
            sub['cycle'],
            y,
            color=color,
            ls=protocol_linestyles.get(protocol, '-'),
            lw=1.8,
            label=label,
        )

        if int(sub['samples'].iloc[0]) > 1:
            lower_signed = signed - err
            upper_signed = signed + err
            lower = np.where(
                (lower_signed <= 0.0) & (upper_signed >= 0.0),
                0.0,
                np.minimum(np.abs(lower_signed), np.abs(upper_signed)),
            )
            upper = np.maximum(np.abs(lower_signed), np.abs(upper_signed))
            ax.fill_between(
                sub['cycle'],
                np.maximum(lower, ENTROPY_EPS),
                np.maximum(upper, ENTROPY_EPS),
                color=color,
                alpha=0.12,
            )
        plotted += 1

if plotted == 0:
    raise RuntimeError(
        f'No matching shifted cross-size curves for nshell={cross_size_nshell}, {cross_size_dw_label}'
    )

ax.set_title(f'Absolute total entropy deviation from 2 log 2 vs cycle, nshell={cross_size_nshell}, {cross_size_dw_label}')
ax.set_xlabel('cycle')
ax.set_ylabel(r'$|S - 2\log 2|/\log 2$')
ax.set_xscale('log')
ax.set_yscale('log')
outside_legend(ax, fontsize=8)

path = FIG_ROOT / f'total_entropy_abs_minus_2log2_cross_size_nsh{cross_size_nshell}_{cross_size_dw_label}_loglog.png'
fig.savefig(path, bbox_inches='tight')
plt.close(fig)
show_saved_image(
    path,
    f'Absolute total entropy deviation from 2 log 2 cross-size comparison, nshell={cross_size_nshell}, {cross_size_dw_label} (log-log)',
)


## 7. Y-Integrated Entropy Profiles

In [ ]:
remove_stale_figures([
    'y_integrated_entropy_fixed_x_lines_by_config.png',
    'y_integrated_entropy_halfspace_contours_by_config.png',
    'y_integrated_cycle100_overlay_by_dwtrunc.png',
    'y_integrated_cycle100_delta_dw1_minus_dw0.png',
    'dw_trunc_y_integrated_cycle100_overlay.png',
    'dw_trunc_y_integrated_cycle100_delta.png',
])
for stale_output in [
    'y_integrated_entropy_spatial_minus_log2_selected_cycles.csv',
    'y_integrated_abs_spatial_minus_log2_selected_cycles.csv',
]:
    stale_path = OUTPUT_ROOT / stale_output
    if stale_path.exists():
        stale_path.unlink()
        print(f'[cleanup] removed stale output {stale_output}')
for pattern in [
    'y_integrated_entropy_spatial_minus_log2_fixed_x_lines_N*.png',
    'y_integrated_abs_spatial_minus_log2_fixed_x_lines_N*.png',
]:
    for stale_path in sorted(FIG_ROOT.glob(pattern)):
        stale_path.unlink()
        print(f'[cleanup] removed stale figure {stale_path.name}')
ordered = sorted(plot_records, key=lambda r: (int(r['summary']['Nx']), int(r['summary']['Ny']), int(r['summary']['nshell']), str(r['summary']['protocol']), r['dw_label']))
for nx, ny in available_system_sizes(ordered):
    system_records = records_for_system(ordered, nx, ny)
    ncols = 2
    nrows = int(np.ceil(len(system_records) / ncols))
    display(Markdown(f'### Y-integrated entropy at fixed x: N{nx}x{ny}'))
    fig, axes = plt.subplots(
        nrows,
        ncols,
        figsize=(8.0 * ncols, max(4.0, 3.5 * nrows)),
        constrained_layout=True,
        sharex=True,
    )
    axes = np.atleast_1d(axes).ravel()
    for ax, rec in zip(axes, system_records):
        s = rec['summary']
        for xi, x in enumerate(rec['x_keep']):
            y = np.maximum(rec['S_yint_half'][:, xi] / ENTROPY_NORM, ENTROPY_EPS)
            ax.plot(rec['cycles'], y, lw=1.0, label=f'x={int(x)}')
        ax.set_title(f"nsh{s['nshell']} {s['protocol']} {rec['dw_label']}")
        ax.set_xlabel('cycle')
        ax.set_ylabel(r'$\sum_y s(x,y)/\ln 2$')
        ax.set_yscale('log')
        outside_legend(ax, fontsize=6, ncol=1)
    for ax in axes[len(system_records):]:
        ax.set_visible(False)
    path = FIG_ROOT / f'y_integrated_entropy_fixed_x_lines_N{nx}x{ny}.png'
    fig.savefig(path, bbox_inches='tight')
    plt.close(fig)
    show_saved_image(path, f'Y-integrated entropy at fixed x, N{nx}x{ny}')

for nx, ny in available_system_sizes(ordered):
    system_records = records_for_system(ordered, nx, ny)
    ncols = 2
    nrows = int(np.ceil(len(system_records) / ncols))
    display(Markdown(f'### Y-integrated absolute spatial entropy deviation from log 2 at fixed x: N{nx}x{ny}'))
    fig, axes = plt.subplots(
        nrows,
        ncols,
        figsize=(8.0 * ncols, max(4.0, 3.5 * nrows)),
        constrained_layout=True,
        sharex=True,
    )
    axes = np.atleast_1d(axes).ravel()
    for ax, rec in zip(axes, system_records):
        s = rec['summary']
        for xi, x in enumerate(rec['x_keep']):
            y = np.abs(rec['S_yint'][:, int(x)] / ENTROPY_NORM - SPATIAL_ENTROPY_OFFSET_OVER_LN2)
            y = np.maximum(y, ENTROPY_EPS)
            ax.plot(rec['cycles'], y, lw=1.0, label=f'x={int(x)}')
        ax.set_title(f"nsh{s['nshell']} {s['protocol']} {rec['dw_label']}")
        ax.set_xlabel('cycle')
        ax.set_ylabel(r'$|\sum_y s(x,y)/\ln 2 - 1|$')
        ax.set_yscale('log')
        outside_legend(ax, fontsize=6, ncol=1)
    for ax in axes[len(system_records):]:
        ax.set_visible(False)
    path = FIG_ROOT / f'y_integrated_abs_spatial_minus_log2_fixed_x_lines_N{nx}x{ny}.png'
    fig.savefig(path, bbox_inches='tight')
    plt.close(fig)
    show_saved_image(path, f'Y-integrated absolute spatial entropy deviation from log 2 at fixed x, N{nx}x{ny}')

selected_cycles = [0, 10, 25, 50, 100]
y_integrated_selected_cycles = (
    y_integrated_entropy_curves[y_integrated_entropy_curves['cycle'].isin(selected_cycles)]
    .sort_values(['Nx', 'Ny', 'nshell', 'protocol', 'dw_label', 'cycle', 'x'])
    .reset_index(drop=True)
)
save_csv_atomic(y_integrated_selected_cycles, OUTPUT_ROOT / 'y_integrated_entropy_selected_cycles.csv')
save_csv_atomic(
    y_integrated_selected_cycles[[
        'campaign_id', 'case_id', 'Nx', 'Ny', 'nshell', 'protocol', 'samples',
        'dw_truncation', 'dw_label', 'entropy_units', 'cycle', 'x',
        'y_integrated_abs_spatial_minus_log2_over_ln2',
    ]],
    OUTPUT_ROOT / 'y_integrated_abs_spatial_minus_log2_selected_cycles.csv',
)
display(Markdown('### y-integrated contour values by x and protocol (`S/ln2`)'))
display(y_integrated_selected_cycles)


## 8. Spatial Entropy Contour Movie

In [ ]:
def make_spatial_comparison_video(records: list[dict], fig_root: Path) -> Path:
    require_ffmpeg()
    from matplotlib.animation import FuncAnimation, FFMpegWriter
    from matplotlib.colors import LogNorm

    video_path = fig_root / 'spatial_entropy_contour_evolution_dw_comparison.mp4'
    ordered = sorted(records, key=lambda r: (int(r['summary']['nshell']), str(r['summary']['protocol']), r['dw_label']))
    cycles = ordered[0]['cycles']
    contour_arrays = [np.maximum(rec['S_xy'] / ENTROPY_NORM, ENTROPY_EPS) for rec in ordered]
    vmax = max(float(np.max(arr)) for arr in contour_arrays)
    positive_min = min(float(np.min(arr[arr > 0.0])) for arr in contour_arrays if np.any(arr > 0.0))
    vmin = max(positive_min, ENTROPY_EPS)
    norm = LogNorm(vmin=vmin, vmax=vmax)

    fig, axes = plt.subplots(2, 4, figsize=(14.0, 7.0), constrained_layout=True, sharex=True, sharey=True)
    axes = axes.ravel()
    images = []
    for ax, rec, contour in zip(axes, ordered, contour_arrays):
        s = rec['summary']
        im = ax.imshow(
            contour[0],
            origin='lower',
            aspect='auto',
            extent=[-0.5, int(s['Ny']) - 0.5, -0.5, int(s['Nx']) - 0.5],
            cmap='Blues',
            norm=norm,
        )
        ax.set_xlabel('y')
        ax.set_ylabel('x')
        ax.text(
            0.02,
            0.96,
            f"nsh{s['nshell']} {s['protocol']} {rec['dw_label']}",
            transform=ax.transAxes,
            ha='left',
            va='top',
            fontsize=8,
            bbox={'facecolor': 'white', 'edgecolor': 'none', 'alpha': 0.75, 'pad': 2},
        )
        images.append(im)
    for ax in axes[len(ordered):]:
        ax.set_visible(False)
    cbar = fig.colorbar(images[0], ax=axes[:len(ordered)].tolist(), shrink=0.86)
    cbar.set_label(r'$s(x,y)/\ln 2$')
    fig.suptitle(f'cycle {int(cycles[0])}')

    def update(frame_idx):
        cycle = int(cycles[frame_idx])
        for im, contour in zip(images, contour_arrays):
            im.set_data(contour[frame_idx])
        fig.suptitle(f'cycle {cycle}')
        return images

    anim = FuncAnimation(fig, update, frames=len(cycles), interval=220, blit=False)
    writer = FFMpegWriter(fps=5, bitrate=2200)
    anim.save(video_path, writer=writer, dpi=150)
    plt.close(fig)
    return video_path

def make_spatial_abs_minus_log2_comparison_video(records: list[dict], fig_root: Path) -> Path:
    require_ffmpeg()
    from matplotlib.animation import FuncAnimation, FFMpegWriter
    from matplotlib.colors import LogNorm

    video_path = fig_root / 'spatial_entropy_abs_minus_log2_contour_evolution_dw_comparison.mp4'
    ordered = sorted(records, key=lambda r: (int(r['summary']['nshell']), str(r['summary']['protocol']), r['dw_label']))
    cycles = ordered[0]['cycles']
    contour_arrays = [
        np.maximum(np.abs(rec['S_xy'] / ENTROPY_NORM - SPATIAL_ENTROPY_OFFSET_OVER_LN2), ENTROPY_EPS)
        for rec in ordered
    ]
    vmax = max(float(np.max(arr)) for arr in contour_arrays)
    positive_min = min(float(np.min(arr[arr > 0.0])) for arr in contour_arrays if np.any(arr > 0.0))
    vmin = max(positive_min, ENTROPY_EPS)
    norm = LogNorm(vmin=vmin, vmax=vmax)

    fig, axes = plt.subplots(2, 4, figsize=(14.0, 7.0), constrained_layout=True, sharex=True, sharey=True)
    axes = axes.ravel()
    images = []
    for ax, rec, contour in zip(axes, ordered, contour_arrays):
        s = rec['summary']
        im = ax.imshow(
            contour[0],
            origin='lower',
            aspect='auto',
            extent=[-0.5, int(s['Ny']) - 0.5, -0.5, int(s['Nx']) - 0.5],
            cmap='magma',
            norm=norm,
        )
        ax.set_xlabel('y')
        ax.set_ylabel('x')
        ax.text(
            0.02,
            0.96,
            f"nsh{s['nshell']} {s['protocol']} {rec['dw_label']}",
            transform=ax.transAxes,
            ha='left',
            va='top',
            fontsize=8,
            bbox={'facecolor': 'white', 'edgecolor': 'none', 'alpha': 0.75, 'pad': 2},
        )
        images.append(im)
    for ax in axes[len(ordered):]:
        ax.set_visible(False)
    cbar = fig.colorbar(images[0], ax=axes[:len(ordered)].tolist(), shrink=0.86)
    cbar.set_label(r'$|s(x,y)/\ln 2 - 1|$')
    fig.suptitle(f'cycle {int(cycles[0])}')

    def update(frame_idx):
        cycle = int(cycles[frame_idx])
        for im, contour in zip(images, contour_arrays):
            im.set_data(contour[frame_idx])
        fig.suptitle(f'cycle {cycle}')
        return images

    anim = FuncAnimation(fig, update, frames=len(cycles), interval=220, blit=False)
    writer = FFMpegWriter(fps=5, bitrate=2200)
    anim.save(video_path, writer=writer, dpi=150)
    plt.close(fig)
    return video_path

for stale_name in ['spatial_entropy_minus_log2_contour_evolution_dw_comparison.mp4']:
    stale_path = FIG_ROOT / stale_name
    if stale_path.exists():
        stale_path.unlink()
        print(f'[cleanup] removed stale video {stale_name}')

if matched_records:
    video_path = make_spatial_comparison_video(matched_records, FIG_ROOT)
    show_saved_video(video_path, 'Spatial entropy contour evolution for matched N16x32 campaigns (`S/ln2`, log color scale)')
    video_path_spatial_abs_minus_log2 = make_spatial_abs_minus_log2_comparison_video(matched_records, FIG_ROOT)
    show_saved_video(
        video_path_spatial_abs_minus_log2,
        'Spatial entropy contour evolution for matched N16x32 campaigns (`|s/ln2 - 1|`, log color scale)',
    )
else:
    video_path = None
    video_path_spatial_abs_minus_log2 = None
    print('[video] skipped comparison movie because no matched N16x32 records were loaded')


## 9. Output Manifest

In [ ]:
output_files = sorted(p.name for p in OUTPUT_ROOT.iterdir() if p.is_file())
figure_files = sorted(p.name for p in FIG_ROOT.iterdir() if p.is_file())
expected_outputs = {
    'run_index.csv',
    'total_entropy_summary.csv',
    'total_entropy_curves.csv',
    'y_integrated_entropy_curves.csv',
    'y_integrated_entropy_selected_cycles.csv',
    'y_integrated_abs_spatial_minus_log2_selected_cycles.csv',
    'validation_report.json',
}
missing = sorted(expected_outputs - set(output_files))
if missing:
    raise RuntimeError(f'Missing expected outputs: {missing}')

for removed_name in [
    'y_integrated_entropy_halfspace_contours_by_config.png',
    'y_integrated_cycle100_overlay_by_dwtrunc.png',
    'y_integrated_cycle100_delta_dw1_minus_dw0.png',
    'dw_trunc_y_integrated_cycle100_overlay.png',
    'dw_trunc_y_integrated_cycle100_delta.png',
    'dw_trunc_total_entropy_overlay_and_delta.png',
    'total_entropy_delta_dw1_minus_dw0.png',
    'total_entropy_delta_dw1_minus_dw0_N16x32.png',
]:
    if removed_name in figure_files:
        raise RuntimeError(f'Removed figure still present: {removed_name}')
for removed_output in [
    'dw_trunc_total_entropy_delta.csv',
    'dw_trunc_y_integrated_entropy_delta.csv',
    'dw_trunc_total_entropy_curves.csv',
    'dw_trunc_total_entropy_summary.csv',
    'dw_trunc_y_integrated_entropy_curves.csv',
    'comparison_manifest.json',
]:
    if removed_output in output_files:
        raise RuntimeError(f'Removed difference output still present: {removed_output}')

analysis_manifest = {
    'analysis_name': 'purification_entropy_contours_maxmix_cpu_observable_first',
    'source_campaigns': [manifest.get('campaign_id') for manifest in all_manifests],
    'source_manifest_paths': [rel_to_bundle(path) for path in manifest_paths],
    'output_root': rel_to_bundle(OUTPUT_ROOT),
    'figures_root': rel_to_bundle(FIG_ROOT),
    'run_count': int(len(run_index)),
    'entropy_units': ENTROPY_UNITS,
    'system_size_grouping': [system_label(nx, ny) for nx, ny in available_system_sizes(records)],
    'outputs': output_files,
    'figures': figure_files,
    'video': None if video_path is None else rel_to_bundle(video_path),
    'spatial_abs_minus_log2_video': None if video_path_spatial_abs_minus_log2 is None else rel_to_bundle(video_path_spatial_abs_minus_log2),
    'notes': [
        'Default report includes the two N16x32_C100 campaigns and the single-nshell N30x40_C100 campaign.',
        'Entropy CSV columns are normalized as S/ln2; raw source NPZ files remain in natural-log units.',
        'Figures are separated by system size; hyperparameter variants within a system size are grouped as subplots/curves.',
        'No dw_trunc1 - dw_trunc0 difference plots or difference CSV products are generated.',
        'The spatial entropy-contour MP4 uses a logarithmic colorbar normalization with a positive entropy floor.',
        'Absolute shifted total entropy columns and figures report |S - 2 log 2|/log 2 on log-log axes.',
        'Absolute shifted spatial entropy-contour MP4 reports |s(x,y)/log 2 - 1| per site; y-integrated shifted curves report |sum_y s(x,y)/log 2 - 1| after integrating over y.',
        'The absolute shifted spatial entropy-contour MP4 uses a logarithmic colorbar normalization with a positive deviation floor.',
        'The selected-cycle y-integrated table reports direct contour values sum_y s(x,y)/ln2 by x, protocol, nshell, and dw_truncation label.',
        'Y-integrated heatmaps and final-cycle y-integrated overlay/delta plots are intentionally not generated.',
        'Stacked derived_arrays.npz is written only when all loaded campaigns share compatible cycle/x grids.',
    ],
}
write_json_atomic(OUTPUT_ROOT / 'analysis_manifest.json', analysis_manifest)

display(Markdown('### Output files'))
display(pd.DataFrame({'outputs': sorted(p.name for p in OUTPUT_ROOT.iterdir() if p.is_file()) + [f'figures/{name}' for name in figure_files]}))
display(Markdown(f'Analysis output folder: `{rel_to_bundle(OUTPUT_ROOT)}`'))
print(f'[done] wrote manifest to {OUTPUT_ROOT / "analysis_manifest.json"}')
